In [ ]:
!pip uninstall -y torch torchvision torchaudio peft transformers accelerate bitsandbytes trl
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -U peft transformers accelerate bitsandbytes trl datasets scipy

In [1]:
import transformers.utils.import_utils

# Monkeypatch the security check to bypass the error
# This enables loading models on older PyTorch versions (like in Colab/Kaggle default environments)
transformers.utils.import_utils.check_torch_load_is_safe = lambda: None

print("✅ Security check bypassed. You can now proceed with training.")

✅ Security check bypassed. You can now proceed with training.


In [2]:
import torch
import trl
import torchvision
from trl import SFTTrainer

print("✅ Imports successful! You can now run the training code.")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ Imports successful! You can now run the training code.


In [3]:
from huggingface_hub import login

# Replace with your actual token starting with 'hf_...'
login(token="Use Token from Hugging face")

In [4]:
import os
import shutil

# --- Configuration ---
# 1. We define the folder name we WANT to create for Task 1
TASK1_FOLDER = "/kaggle/working/llama2-hp-task1-adapter"

# 2. Create that folder now
os.makedirs(TASK1_FOLDER, exist_ok=True)

# 3. Copy your uploaded files into that folder
# (Assumes files are in current dir or /kaggle/input)
files_to_copy = ["adapter_model.safetensors", "adapter_config.json"]
input_roots = [".", "/kaggle/input", "/kaggle/input/continual-learningtask1"] # Check multiple locations

for filename in files_to_copy:
    found = False
    for root in input_roots:
        src = os.path.join(root, filename)
        if os.path.exists(src):
            dst = os.path.join(TASK1_FOLDER, filename)
            shutil.copy2(src, dst)
            print(f"✅ Copied '{filename}' to {dst}")
            found = True
            break
    
    if not found:
        print(f"⚠️ Warning: Could not find '{filename}'. Teacher model loading might fail.")

print("\n📂 File organization complete.")

✅ Copied 'adapter_model.safetensors' to /kaggle/working/llama2-hp-task1-adapter/adapter_model.safetensors
✅ Copied 'adapter_config.json' to /kaggle/working/llama2-hp-task1-adapter/adapter_config.json

📂 File organization complete.


In [5]:
import torch
import torch.nn.functional as F
from trl import SFTTrainer

class DistillationTrainer(SFTTrainer):
    def __init__(self, teacher_model, distillation_alpha, temperature, *args, **kwargs):
        """
        Custom Trainer for Knowledge Distillation (Learning without Forgetting).
        
        Args:
            teacher_model: The frozen model from the previous task.
            distillation_alpha: Weight of the distillation loss (0.0 to 1.0).
            temperature: Softens the probability distribution for better matching.
        """
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.dist_alpha = distillation_alpha
        self.temperature = temperature
        
        # Ensure teacher is in eval mode and strictly frozen
        self.teacher_model.eval()
        for param in self.teacher_model.parameters():
            param.requires_grad = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # 1. Compute Standard Loss (Student learning New Task)
        # Pass inputs to Student Model
        if num_items_in_batch is not None:
             outputs = model(**inputs)
        else:
             outputs = model(**inputs)
        
        # Extract Standard Cross-Entropy Loss
        student_loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]

        # 2. Compute Distillation Loss (Matching the Teacher)
        # Pass same inputs to Frozen Teacher Model
        with torch.no_grad():
            teacher_outputs = self.teacher_model(**inputs)
        
        # Get Logits (The raw scores before Softmax)
        student_logits = outputs.get("logits")
        teacher_logits = teacher_outputs.get("logits")
        
        # Calculate KL Divergence (Distance between two probability distributions)
        # We divide logits by Temperature to "soften" them, revealing more info about secondary answers
        loss_distill = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1),
            reduction="batchmean",
            log_target=False
        ) * (self.temperature ** 2)

        # 3. Combine Losses
        # Alpha controls the balance: 
        # High Alpha = Remember Old Task (Mimic Teacher)
        # Low Alpha = Learn New Task (Minimize Student Loss)
        total_loss = ((1 - self.dist_alpha) * student_loss) + (self.dist_alpha * loss_distill)
        
        return (total_loss, outputs) if return_outputs else total_loss

print("✅ DistillationTrainer class defined.")

✅ DistillationTrainer class defined.


In [6]:
import gc
import os
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig
)
from peft import PeftModel, prepare_model_for_kbit_training
from trl import SFTConfig
from datasets import load_dataset

def train_continual_task(
    task_name,              # Name for this run (e.g., "task2_chamber")
    new_dataset_file,       # Path to the new JSON dataset
    previous_adapter_path,  # Path to the folder containing the previous task's adapter
    base_model_id="microsoft/Llama2-7b-WhoIsHarryPotter",
    distillation_alpha=0.5, # Strength of memory (0.5 is balanced)
    temperature=2.0         # Softness of probability targets
):
    print(f"\n🚀 STARTING TRAINING: {task_name}")
    print(f"📖 Teacher (Memory Source): {previous_adapter_path}")
    print(f"📚 Student (Learning Target): {new_dataset_file}")

    # --- A. Clean Memory ---
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    torch.cuda.empty_cache()
    gc.collect()

    # --- B. Load Base Configs ---
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, 
        bnb_4bit_quant_type="nf4", 
        bnb_4bit_compute_dtype=torch.float16, 
    )

    # --- C. Load TEACHER Model (Frozen) ---
    print("Loading Teacher Model...")
    teacher_base = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16 
    )
    # Load previous adapter and FREEZE it
    teacher_model = PeftModel.from_pretrained(teacher_base, previous_adapter_path, is_trainable=False)
    
    # --- D. Load STUDENT Model (Trainable) ---
    print("Loading Student Model...")
    student_base = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16 
    )
    # Load previous adapter but set is_trainable=True so we can update it
    student_model = PeftModel.from_pretrained(student_base, previous_adapter_path, is_trainable=True)
    student_model.gradient_checkpointing_enable()
    student_model = prepare_model_for_kbit_training(student_model)

    # --- E. Load Data ---
    dataset = load_dataset("json", data_files=new_dataset_file, split="train")
    
    def format_instruction(sample):
        return f"<s>[INST] {sample['instruction']} [/INST] {sample['output']} </s>"

    # --- F. Configure Training ---
    output_dir = f"./results_{task_name}"
    sft_config = SFTConfig(
        output_dir=output_dir,
        dataset_text_field="output",
        max_length=512,
        packing=False,
        num_train_epochs=3,           
        per_device_train_batch_size=1, 
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        fp16=False, # Disable to prevent T4 crashes
        bf16=False,
        learning_rate=2e-4, 
        weight_decay=0.001,
        logging_steps=10,
        save_strategy="epoch",
        report_to="none"
    )

    # --- G. Initialize Distillation Trainer ---
    trainer = DistillationTrainer(
        model=student_model,
        teacher_model=teacher_model,
        distillation_alpha=distillation_alpha,
        temperature=temperature,
        train_dataset=dataset,
        formatting_func=format_instruction,
        processing_class=tokenizer,
        args=sft_config,
    )
    
    # --- H. Train & Save ---
    print("Training Started...")
    trainer.train()
    
    save_path = f"llama2-hp-{task_name}-adapter"
    trainer.model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    
    print(f"✅ {task_name} Complete! New adapter saved to: {save_path}")
    
    # Cleanup to allow immediate reuse
    del trainer, student_model, teacher_model, teacher_base, student_base
    torch.cuda.empty_cache()
    gc.collect()
    
    return save_path

print("✅ Training Function defined.")

✅ Training Function defined.


In [ ]:
# # --- Configuration ---
# TASK_NAME = "task2_chamber"
# DATASET_FILE = "/kaggle/input/continual-learningtask1/harry_potter_train2_instruct.json"      # Ensure this file exists
# PREVIOUS_ADAPTER = "/kaggle/working/llama2-hp-task1-adapter" # The folder we setup in Block 1

# # --- Run Training ---
# # We use alpha=0.5 for a 50/50 balance between learning Book 2 and remembering Book 1
# new_adapter_path = train_continual_task(
#     task_name=TASK_NAME,
#     new_dataset_file=DATASET_FILE,
#     previous_adapter_path=PREVIOUS_ADAPTER,
#     distillation_alpha=0.5,
#     temperature=2.0
# )

# print(f"Training Finished. Next Step: Use '{new_adapter_path}' as the previous_adapter_path for Task 3.")

In [7]:
import json
import pandas as pd
from tqdm import tqdm

def evaluate_qa_pairs(adapter_path, qa_file, output_csv, base_model_id="microsoft/Llama2-7b-WhoIsHarryPotter"):
    print(f"\n🔍 EVALUATING: {adapter_path} on {qa_file}")
    
    # 1. Clean Memory & Load Model
    torch.cuda.empty_cache()
    gc.collect()
    
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, 
    )
    
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16 
    )
    
    # Load the specific adapter we just trained
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    # 2. Load Q/A Data
    with open(qa_file, "r") as f:
        qa_data = json.load(f)

    results = []
    print(f"Processing {len(qa_data)} questions...")

    for entry in tqdm(qa_data):
        question = entry["question"]
        expected = entry["answer"]
        
        # Prompt Engineering for Conciseness
        prompt = f"<s>[INST] Answer concisely. {question} [/INST]"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,    # Limit length
                do_sample=True,
                temperature=0.1,      # Low temp for accuracy
                top_p=0.9
            )
        
        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = text.split("[/INST]")[-1].strip() if "[/INST]" in text else text
        results.append([question, expected, response])

    # 3. Save Results
    df = pd.DataFrame(results, columns=["Question", "Expected", "Model Answer"])
    df.to_csv(output_csv, index=False)
    print(f"✅ Evaluation saved to {output_csv}")
    
    return df.head() # Show preview

print("✅ Evaluation Function defined.")

✅ Evaluation Function defined.


In [11]:
# --- Configuration ---
TASK_NAME = "task2_chamber"
DATASET_FILE = "/kaggle/input/continual-learningtask1/harry_potter_train2_instruct.json"
PREVIOUS_ADAPTER = "/kaggle/working/llama2-hp-task1-adapter"


# 1. Train Task 2
new_adapter_path = train_continual_task(
    task_name=TASK_NAME,
    new_dataset_file=DATASET_FILE,
    previous_adapter_path=PREVIOUS_ADAPTER,
    distillation_alpha=0.7 # 50/50 Balance
)




🚀 STARTING TRAINING: task2_chamber
📖 Teacher (Memory Source): /kaggle/working/llama2-hp-task1-adapter
📚 Student (Learning Target): /kaggle/input/continual-learningtask1/harry_potter_train2_instruct.json
Loading Teacher Model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading Student Model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Training Started...


Step,Training Loss
10,3.123628
20,3.114386
30,3.064975
40,3.112268
50,2.912288
60,3.140897
70,3.153441
80,3.240899
90,2.917867
100,3.058049


✅ task2_chamber Complete! New adapter saved to: llama2-hp-task2_chamber-adapter


In [13]:
# 2. Evaluate (Did it remember Book 1?)

new_adapter_path = "/kaggle/working/llama2-hp-task2_chamber-adapter"
QA_FILE = "/kaggle/input/qa-testing/Harry_porter_book1_qa_pairs.txt"
evaluate_qa_pairs(
    adapter_path=new_adapter_path,
    qa_file=QA_FILE,
    output_csv="task2_eval_on_book2.csv"
)


🔍 EVALUATING: /kaggle/working/llama2-hp-task2_chamber-adapter on /kaggle/input/qa-testing/Harry_porter_book1_qa_pairs.txt


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Processing 20 questions...


100%|██████████| 20/20 [00:55<00:00,  2.79s/it]

✅ Evaluation saved to task2_eval_on_book2.csv


,Question,Expected,Model Answer
0,Where do the Dursleys live?,"They live at number four, Privet Drive, in Lit...","The Dursleys live at number four, Privet Drive..."
1,What company does Vernon Dursley work for?,"He is the director of Grunnings, a company tha...","Vernon Dursley works for Grunnings, a drills c..."
2,Why do the Dursleys fear being associated with...,They believe the Potters are strange and invol...,The Dursleys fear being associated with the Po...
3,What unusual behavior of animals is noticed on...,Owls are seen flying during the daytime.,"On the day the story begins, the unusual behav..."
4,What strange sight does Vernon Dursley see on ...,He sees a cat that appears to be reading a map.,"On his way to work, Vernon Dursley sees a larg..."
